# Environmental and Biological Measurements from Estuarine and Coastal Sampling Stations in the Basque Country, Bay of Biscay (1995–2014) Exploration with `mlcroissant`
This notebook provides a step-by-step demonstration for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset is described by a Croissant schema at:
https://sen.science/doi/10.71728/senscience.tpgp-31sg/fair2.json


In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.tpgp-31sg/fair2.json'

# Load the dataset object
dataset = mlc.Dataset(croissant_url)

# Access metadata
metadata = dataset.metadata

print("Dataset Name: {}".format(metadata.name))
print("Description: {}".format(metadata.description))

# print additional metadata summary
print(f"Identifier: {metadata.identifier}")
print(f"License: {metadata.license}")
print(f"Spatial Coverage: {metadata.spatialCoverage}")
print(f"Temporal Coverage: {metadata.temporalCoverage}")

## 2. Data Overview
Review available record sets and their fields using `@id` references.

In [ ]:
# List all record sets defined in the dataset
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = []
    for record_set in metadata.recordSet:
        # The record_set should have '@id' and potentially 'field' or 'column'
        print(f"Record Set @id: {record_set['@id']}")
        record_sets.append(record_set['@id'])
        # List its fields
        if 'field' in record_set:
            fields = record_set['field']
            field_ids = [f['@id'] for f in fields]
            print(f"  Fields: {field_ids}")
        elif 'column' in record_set:
            column_ids = [c['@id'] for c in record_set['column']]
            print(f"  Columns: {column_ids}")
        else:
            print("  No fields or columns listed.")
else:
    print("No record sets are defined in metadata. Please check the schema.")

## 3. Data Extraction
Load records from a specific record set into a DataFrame for analysis.
All references use `@id` fields for clarity and consistency.

In [ ]:
# If record sets are defined, extract data for analysis

# Demo: Use the first available record set (by @id), or create an example with a placeholder
dataframes = {}
available_record_sets = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    # collect record set @ids
    for rs in metadata.recordSet:
        available_record_sets.append(rs['@id'])
else:
    # Example: If not in metadata, try to discover record sets using dataset.schema
    schema = dataset._schema
    if 'recordSet' in schema:
        for rs in schema['recordSet']:
            available_record_sets.append(rs['@id'])

print("Record Set @ids:", available_record_sets)

# Choose first valid record set for demonstration
if available_record_sets:
    sample_record_set_id = available_record_sets[0]
else:
    sample_record_set_id = None

# Load records from each record set
for record_set_id in available_record_sets:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for Record Set {record_set_id}: {df.columns.tolist()}")
    print(df.head())

# Show columns from sample record set
if sample_record_set_id:
    print(f"Sample Data Columns ({sample_record_set_id}):")
    print(dataframes[sample_record_set_id].columns.tolist())
    display(dataframes[sample_record_set_id].head())
else:
    print("No record sets found for extraction.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps such as filtering, normalizing, or grouping records based on specific criteria.
Reference all fields by their `@id` values.

In [ ]:
# Select a numeric field for analysis
# Assumption: At least one record set and field with numeric data is available
if sample_record_set_id:
    df = dataframes[sample_record_set_id]
    # Try to automatically choose a numeric field using dtypes
    numeric_field_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
    if numeric_field_candidates:
        numeric_field_id = numeric_field_candidates[0]
        threshold = df[numeric_field_id].mean() if not df[numeric_field_id].isnull().all() else 10
        try:
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            print(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Grouping
            # Try to group by a categorical field
            group_field_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col])]
            if group_field_candidates:
                group_field = group_field_candidates[0]
                grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
                print(f"Grouped data by {group_field}:")
                print(grouped_df.head())
            else:
                print("No categorical field found for grouping.")
        except Exception as e:
            print("EDA Error:", e)
    else:
        print("No numeric fields found in sample record set.")
else:
    print("No record set available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Visualize the distribution of the numeric field in filtered records
import matplotlib.pyplot as plt
import seaborn as sns

if sample_record_set_id and 'filtered_df' in locals() and not filtered_df.empty:
    plt.figure(figsize=(8, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} in Filtered Records")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping is available, show distribution by group
    if 'group_field' in locals():
        plt.figure(figsize=(10, 5))
        sns.boxplot(x=filtered_df[group_field], y=filtered_df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Loaded dataset metadata for environmental and biological measurements covering Basque Country coastal and estuarine stations (1995–2014).
- Reviewed available record sets and extracted sample records using their `@id` values.
- Performed basic EDA: filtered and normalized numeric fields, grouped data by categories, and visualized distributions.
- The dataset is suitable for longitudinal environmental trend analysis, supporting research and regulatory reporting.